In [31]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
SRC = ROOT / "src"

sys.path.insert(0, str(SRC))

from scipy.stats import norm
import numpy as np
import pandas as pd

from utils.data_generation import generate_two_estimator_data
from utils.coverage import psi_two_estimator, ci_delta_two_estimator, ci_tau_two_estimator

In [45]:
B = 5000
tau = 5
delta = 0.1
sigma1 = 10
sigma2 = 10
rho = 0
n_1 = 50
n_2 = 10000
alpha = 0.2
gamma = 0.1
rng = np.random.default_rng()

# Record lower and upper bounds of the confidence intervals.
ci_eb_array = np.zeros((B, 2))
ci_regular_array = np.zeros((B, 2))
estimates_eb_array = np.zeros(B)
estimates_regular_array = np.zeros(B)

for i in range(B):
    y, _, _, _ = generate_two_estimator_data(
        n1=n_1, n2=n_2, mu=tau, delta_f=delta,
        sigma1=sigma1, sigma2=sigma2, rho=rho, rng=rng,
        return_samples=True,
    )

    estimate_eb = y[0] + psi_two_estimator(y, n1=n_1, n2=n_2, sigma1=sigma1, sigma2=sigma2, rho=rho) * (y[1] - y[0])
    estimates_eb_array[i] = estimate_eb
    estimates_regular_array[i] = y[0]

    ci_eb = ci_tau_two_estimator(
        y, n1=n_1, n2=n_2, sigma1=sigma1, sigma2=sigma2,
        rho=rho, alpha=alpha, gamma=gamma,
    )
    # Use the regular normal interval for the first estimator as a comparison.
    z = norm.ppf(1 - gamma / 2)
    ci_regular = (y[0] - z * sigma1 / np.sqrt(n_1), y[0] + z * sigma1 / np.sqrt(n_1))

    ci_eb_array[i, :] = ci_eb
    ci_regular_array[i, :] = ci_regular


In [46]:
mse_eb = np.mean((estimates_eb_array - tau) ** 2)
mse_regular = np.mean((estimates_regular_array - tau) ** 2)

mse_eb, mse_regular

(np.float64(0.7080228162564356), np.float64(2.0497434856527494))

In [47]:
# check coverage of the confidence intervals
coverage_eb = np.mean((ci_eb_array[:, 0] <= tau) & (ci_eb_array[:, 1] >= tau))
coverage_regular = np.mean((ci_regular_array[:, 0] <= tau) & (ci_regular_array[:, 1] >= tau))

avg_length_eb = np.mean(ci_eb_array[:, 1] - ci_eb_array[:, 0])
avg_length_regular = np.mean(ci_regular_array[:, 1] - ci_regular_array[:, 0])

In [48]:
coverage_eb, coverage_regular

(np.float64(0.986), np.float64(0.8964))

In [49]:
avg_length_eb, avg_length_regular

(np.float64(6.329000944526393), np.float64(4.652348614706694))